In [ ]:
### This workflow performs the following functions. Differential expression analysis which can be done per genotype, per condition, per cluster etc.
### Compositional analysis to investigate changes in cell type/cluster/cell cycle state changes.
### MAGIC imputation for improved smoothing and visualisation.
### Signature scoring of known gene sets and custom gene sets. E.g. Biological pathways, cancer hallmarks, custom DEGs from bulk RNAseq or ATACseq.

### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(40)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 40)
library(SeuratDisk)
library(reshape2)
library(tidyverse)
library(RColorBrewer)
library(tidyr)
library(grid)
library(cowplot)
library(clusterProfiler)
library(msigdbr)
library(gprofiler2)
library(scCustomize)
library(reticulate)
library(ggrepel)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/2025_2026_int/"
repDir <- paste0(mainDir, "composition_DEG_signatures/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- "/data/ebaird/scRNAseq/refs/"


dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)
dir.create(refsDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan", "magenta", "deeppink4")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

best_colors <- viridis(20)
best_colors_magma <- magma(20)
best_colors_plasma <- plasma(20)

In [ ]:
### Load object
seu <- readRDS(paste0(mainDir, "QC_clustering/integrated.rds"))

In [ ]:
# Get unique genotypes as subsets
seu$genotype <- as.character(seu$genotype)
samples <- unique(as.character(seu$genotype))

seurat_subsets <- lapply(samples, function(g) {
  subset(seu, cells = rownames(seu@meta.data[seu$genotype == g, ]))
})

names(seurat_subsets) <- samples
seurat_subsets


In [ ]:
### Differential gene expression analysis

DE_dir <- paste0(figDir, "DE_analysis/")
dir.create(DE_dir, recursive = TRUE, showWarnings = FALSE)

# Perform differential expression analysis for all genotype pairs
de_results_list <- list()
filtered_results_list <- list()
top_genes_list <- list()

genotypes <- unique(seu$genotype)

for (i in 1:(length(genotypes)-1)) {
  for (j in (i+1):length(genotypes)) {
    g1 <- genotypes[i]
    g2 <- genotypes[j]
    pair_name <- paste0(g1, "_vs_", g2)
    message("Comparing: ", g1, " vs ", g2)
    de_results <- FindMarkers(
      object = seu,
      ident.1 = g1,
      ident.2 = g2,
      group.by = "genotype",
      min.pct = 0.25,
      logfc.threshold = 0.25
    )
    de_results_list[[pair_name]] <- de_results
    filtered_results <- subset(de_results, p_val < 0.05 & abs(avg_log2FC) > 0.5)
    filtered_results_list[[pair_name]] <- filtered_results
    write.csv(filtered_results, file.path(tabDir, paste0("DE_results_", pair_name, ".csv")))
    top_genes <- head(rownames(filtered_results[order(-abs(filtered_results$avg_log2FC)), ]), 15)
    top_genes_list[[pair_name]] <- top_genes

    # Combined violin plot for this pair
    if (length(top_genes) > 0) {
      expr_data <- FetchData(seu, vars = c(top_genes, "genotype"))
      expr_data_long <- pivot_longer(expr_data, cols = all_of(top_genes), names_to = "gene", values_to = "expression")
      expr_data_long$genotype <- factor(expr_data_long$genotype, levels = c(g1, g2))
      vln_genotype <- ggplot(expr_data_long, aes(x = gene, y = expression, fill = genotype)) +
        geom_violin(position = position_dodge(0.8), width = 0.7, scale = "width", trim = TRUE) +
        stat_summary(fun = median, geom = "point", position = position_dodge(0.8), size = 0.8, color = "black") +
        theme_minimal() +
        labs(title = paste("Top 15 DE Genes:", g1, "vs", g2), x = "Gene", y = "Expression") +
        theme(
          plot.title = element_text(hjust = 0.5, size = 14),
          axis.text.x = element_text(angle = 45, hjust = 1),
          legend.title = element_blank()
        )
      jpeg(file.path(DE_dir, paste0("DE_violin_", pair_name, ".jpeg")), width = 1500, height = 700, res = 150)
      print(vln_genotype)
      dev.off()
    }
  }
}

In [ ]:
### Differential expression per cluster for genotype and timepoint with safe checks

genotypes <- unique(seu$genotype)

safe_find_markers <- function(obj,
                              group.by,
                              ident.1,
                              ident.2 = NULL,
                              min.cells.group = 3,
                              ...) {
  if (!(group.by %in% colnames(obj@meta.data))) {
    stop("group.by column not found in meta.data: ", group.by)
  }

  counts <- table(obj@meta.data[[group.by]])

  if (!(ident.1 %in% names(counts))) {
    message(sprintf("Skipping: ident.1 '%s' not present in this subset.", ident.1))
    return(NULL)
  }
  if (!is.null(ident.2) && !(ident.2 %in% names(counts))) {
    message(sprintf("Skipping: ident.2 '%s' not present in this subset.", ident.2))
    return(NULL)
  }

  if (counts[[ident.1]] < min.cells.group) {
    message(sprintf("Skipping: group '%s' has %d cells (< %d).",
                    ident.1, counts[[ident.1]], min.cells.group))
    return(NULL)
  }
  if (!is.null(ident.2) && counts[[ident.2]] < min.cells.group) {
    message(sprintf("Skipping: group '%s' has %d cells (< %d).",
                    ident.2, counts[[ident.2]], min.cells.group))
    return(NULL)
  }

  res <- tryCatch(
    {
      FindMarkers(
        object = obj,
        ident.1 = ident.1,
        ident.2 = ident.2,
        group.by = group.by,
        ...
      )
    },
    error = function(e) {
      message("FindMarkers error: ", e$message)
      return(NULL)
    }
  )

  return(res)
}

create_combined_violin <- function(seu_subset, top_genes, group_var, title_prefix) {
  if (is.null(top_genes) || length(top_genes) == 0) {
    message("No top genes provided for plotting: ", title_prefix)
    return(NULL)
  }

  expr_data <- FetchData(seu_subset, vars = c(top_genes, group_var))
  expr_data_long <- pivot_longer(expr_data, cols = all_of(top_genes), names_to = "gene", values_to = "expression")
  expr_data_long[[group_var]] <- factor(expr_data_long[[group_var]])

  p <- ggplot(expr_data_long, aes(x = gene, y = expression, fill = .data[[group_var]])) +
    geom_violin(position = position_dodge(0.8), width = 0.7, scale = "width", trim = TRUE) +
    stat_summary(fun = median, geom = "point", position = position_dodge(0.8), size = 0.6, color = "black") +
    theme_minimal() +
    labs(title = paste0(title_prefix, ": Top DE Genes by ", group_var), x = "Gene", y = "Expression") +
    theme(
      plot.title = element_text(hjust = 0.5, size = 12),
      axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
      axis.text.y = element_text(size = 8),
      legend.title = element_blank()
    )

  return(p)
}

# --- DE per cluster for genotype/timepoint -----------------
Idents(seu) <- "seurat_clusters"
cluster_ids <- unique(seu$seurat_clusters)

min.cells.group <- 3  

for (cluster_id in cluster_ids) {
  cluster_id_chr <- as.character(cluster_id)
  message("Processing cluster: ", cluster_id_chr)

  cluster_obj <- subset(seu, idents = cluster_id)

  for (i in 1:(length(genotypes)-1)) {
    for (j in (i+1):length(genotypes)) {
      g1 <- genotypes[i]
      g2 <- genotypes[j]
      pair_name <- paste0(g1, "_vs_", g2)

      de_genotype <- safe_find_markers(
        obj = cluster_obj,
        group.by = "genotype",
        ident.1 = g1,
        ident.2 = g2,
        min.cells.group = min.cells.group,
        min.pct = 0.25,
        logfc.threshold = 0.25
      )

      fn_geno <- file.path(tabDir, paste0("DE_results_cluster_", cluster_id_chr, "_", pair_name, ".csv"))
      if (is.null(de_genotype) || nrow(de_genotype) == 0) {
        message("No genotype DE results for cluster ", cluster_id_chr, " (", pair_name, ") — writing empty file.")
        write.csv(data.frame(), fn_geno, row.names = FALSE)
        top_genes_genotype <- character(0)
      } else {
        filtered_genotype <- subset(de_genotype, p_val < 0.05 & abs(avg_log2FC) > 0.5)
        write.csv(filtered_genotype, fn_geno)
        top_genes_genotype <- head(rownames(filtered_genotype[order(-abs(filtered_genotype$avg_log2FC)), , drop = FALSE]), 10)
      }

      vln_genotype <- NULL
      if (length(top_genes_genotype) > 0) {
        vln_genotype <- create_combined_violin(cluster_obj, top_genes_genotype, "genotype", paste("Cluster", cluster_id_chr, pair_name))
      } else {
        message("No top genotype genes to plot for cluster ", cluster_id_chr, " (", pair_name, ")")
      }

      if (!is.null(vln_genotype)) {
        outfn <- file.path(DE_dir, paste0("Combined_DE_violin_cluster_", cluster_id_chr, "_", pair_name, ".jpeg"))
        jpeg(outfn, width = 1500, height = 800, res = 150)
        print(vln_genotype)
        dev.off()
        message("Wrote: ", outfn)
      } else {
        message("Skipping plot for cluster ", cluster_id_chr, " (", pair_name, ") (no violin to draw).")
      }
    }
  }
}

In [ ]:
# Differential expression cluster 10, HmgZ vs all other genotypes
cluster_10 <- subset(seu, idents = "10")

de_cluster_10_genotype <- safe_find_markers(
  obj = cluster_10,
  group.by = "genotype",
  ident.1 = "HmgZ",
  ident.2 = NULL,
  min.cells.group = min.cells.group,
  min.pct = 0.25,
  logfc.threshold = 0.25
)

fn_cluster_10 <- file.path(tabDir, "DE_results_cluster_10_HmgZ_vs_all.csv")
write.csv(de_cluster_10_genotype, fn_cluster_10, row.names = TRUE)

# ---- Violin plot for top DE genes ----
library(Seurat)
library(ggplot2)

# add gene column if needed
de_cluster_10_genotype$gene <- rownames(de_cluster_10_genotype)

# choose top upregulated genes in HmgZ
top_genes_10 <- de_cluster_10_genotype |>
  subset(avg_log2FC > 0) |>
  arrange(desc(avg_log2FC)) |>
  head(6) |>
  pull(gene)

# violin plot
p_cluster_10_vln <- VlnPlot(
  object = cluster_10,
  features = top_genes_10,
  group.by = "genotype",
  pt.size = 0,
  stack = FALSE,
  flip = FALSE
) + NoLegend()

p_cluster_10_vln

# save
ggsave(
  filename = file.path(plotDir, "VlnPlot_cluster_10_HmgZ_vs_all_top_genes.pdf"),
  plot = p_cluster_10_vln,
  width = 14,
  height = 8
)

In [ ]:
VlnPlot(
  object = cluster_10,
  features = "HmgZ",
  group.by = "genotype",
  pt.size = 0
)

In [ ]:
### Create count tables
# count_table_clusters_samples <- table(seu@meta.data$seurat_clusters, seu@meta.data[["timepoint"]])
# write.csv(count_table_clusters_samples, file.path(tabDir, "counts_per_cluster_per_timepoint.csv"))

count_table <- table(seu@meta.data$seurat_clusters, seu@meta.data[["condition"]])
write.csv(count_table, file.path(tabDir, "counts_per_cluster_per_condition.csv"))

count_table <- table(seu@meta.data$seurat_clusters, seu@meta.data[["genotype"]])
write.csv(count_table, file.path(tabDir, "counts_per_cluster_per_genotype.csv"))

In [ ]:
# Table: genotype vs cluster
ta <- melt(t(table(seu$genotype, seu$seurat_clusters)))
colnames(ta) <- c("genotype", "cluster", "ncells")
ta$genotype <- as.factor(ta$genotype)
ta$cluster <- as.factor(ta$cluster)

g <- ggplot(aes(genotype, ncells, fill=cluster), data=ta) +
  geom_bar(stat="identity") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

g1 <- ggplot(aes(cluster, ncells, fill=cluster), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~genotype, ncol=4) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

g2 <- ggplot(aes(genotype, ncells, fill=genotype), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~cluster, ncol=18) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

pdf(paste0(figDir, "barplot_cells_per_cluster_genotype.pdf"), width=10, height=5)
print(g)
print(g1)
print(g2)
dev.off()

write.csv(ta, file=paste0(tabDir, "barplot_cells_per_cluster_genotype.csv"))

# Cell cycle by cluster
ta <- melt(t(table(seu$seurat_clusters, seu$Phase)))
colnames(ta) <- c("cluster", "Phase", "ncells")
ta$Phase <- as.factor(ta$Phase)
ta$cluster <- as.factor(ta$cluster)

g <- ggplot(aes(cluster, ncells, fill=Phase), data=ta) +
  geom_bar(stat="identity") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

g1 <- ggplot(aes(Phase, ncells, fill=Phase), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~cluster, ncol=4) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

g2 <- ggplot(aes(cluster, ncells, fill=cluster), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~Phase, ncol=16) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

pdf(paste0(figDir, "barplot_cells_per_Phase_clust.pdf"), width=10, height=5)
print(g)
print(g1)
print(g2)
dev.off()

write.csv(ta, file=paste0(tabDir, "barplot_cells_per_Phase_cond.csv"))

### Double normalisation of cluster composition for library size and cluster size
ta <- melt(table(seu$genotype, seu$seurat_clusters))
colnames(ta) <- c("genotype", "cluster", "ncells")
ta$genotype <- as.factor(ta$genotype)
ta$cluster <- as.factor(ta$cluster)
# Normalise to percentage within each genotype
ta$ncells_genotype_perc <- ave(ta$ncells, ta$genotype, FUN=function(x) x / sum(x) * 100)
# Normalise to percentage within each cluster
ta$ncells_cluster_perc <- ave(ta$ncells, ta$cluster, FUN=function(x) x / sum(x) * 100)
# Relative to total cells (double normalization)
# Observed proportion of cells per cluster from each genotype / expected proportion of cells per cluster from each genotype if equally distributed
# =1 equal distribution to the expected. >1 more cells in the cluster from the genotype than expected, <1 less cells in the cluster from the genotype than expected
total_cells <- sum(ta$ncells)
ta$composition_normalised_for_library_size <- (ta$ncells / ave(ta$ncells, ta$genotype, FUN=sum)) / 
                         (ave(ta$ncells, ta$cluster, FUN=sum) / total_cells)

write.csv(ta, file=paste0(tabDir, "cells_per_cluster_genotype_multiple_normalizations.csv"))

g <- ggplot(aes(genotype, composition_normalised_for_library_size, fill=genotype), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~cluster, ncol=4) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

pdf(paste0(figDir, "cluster_composition_normalised_for_library_size.pdf"), width=12, height=8)
print(g)
dev.off()

### Double normalised phase composition for library size and cluster size
ta <- melt(table(seu$Phase, seu$seurat_clusters))
colnames(ta) <- c("Phase", "cluster", "ncells")
ta$cluster <- as.factor(ta$cluster)
ta$Phase <- as.factor(ta$Phase)
# Normalise to percentage within each genotype
ta$ncells_cluster_perc <- ave(ta$ncells, ta$cluster, FUN=function(x) x / sum(x) * 100)
# Normalise to percentage within each Phase
ta$ncells_phase_perc <- ave(ta$ncells, ta$Phase, FUN=function(x) x / sum(x) * 100)
# Relative to total cells (double normalization)
# Observed proportion of cells per Phase from each genotype / expected proportion of cells per Phase from each genotype if equally distributed
# =1 equal distribution to the expected. >1 more cells in the Phase from the genotype than expected, <1 less cells in the Phase from the genotype than expected
total_cells <- sum(ta$ncells)
ta$composition_normalised_for_library_size <- (ta$ncells / ave(ta$ncells, ta$cluster, FUN=sum)) / 
                         (ave(ta$ncells, ta$Phase, FUN=sum) / total_cells)
                        
write.csv(ta, file=paste0(tabDir, "cells_per_Phase_genotype_multiple_normalizations.csv"))
g <- ggplot(aes(Phase, composition_normalised_for_library_size, fill=Phase), data=ta) +
  geom_bar(stat="identity") +
  facet_wrap(~cluster, ncol=4) +
  scale_fill_manual(values = mycols20) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust=1, size = 11, color=1))

pdf(paste0(figDir, "Phase_composition_cluster_normalised_for_library_size.pdf"), width=12, height=8)
print(g)
dev.off()

In [ ]:
create_balanced_umap_hybrid <- function(seurat_obj, 
                                       genotype_var = "condition",
                                       cluster_var = "seurat_clusters", 
                                       jitter_amount = 0.1,
                                       min_cells_per_cluster = 10) {
  
  reduction_name <- "umap"
  cat("Using reduction:", reduction_name, "\n")
  
  coords <- as.data.frame(seurat_obj@reductions[[reduction_name]]@cell.embeddings[, 1:2])
  colnames(coords) <- c("UMAP_1", "UMAP_2")
  
  coords[[genotype_var]] <- seurat_obj@meta.data[[genotype_var]]
  coords[[cluster_var]] <- seurat_obj@meta.data[[cluster_var]]
  coords$cell_id <- rownames(coords)
  
  coords$sampling_method <- "real"
  
  genotype_counts <- table(coords[[genotype_var]])
  max_size <- max(genotype_counts)
  cat("Largest library has", max_size, "cells\n")
  largest_lib <- names(genotype_counts)[which.max(genotype_counts)]
  cat("Largest library is:", largest_lib, "\n")

  balanced_umap_list <- list(coords)
  
  for (genotype in names(genotype_counts)) {
    current_size <- genotype_counts[genotype]
    cat("Processing", genotype, "with", current_size, "cells\n")
    
    if (current_size < max_size) {
      real_cells <- coords[coords[[genotype_var]] == genotype, ]
      n_artificial <- max_size - current_size
      cluster_props <- table(real_cells[[cluster_var]]) / nrow(real_cells)
      artificial_cells_list <- list()
      
      for (cluster in names(cluster_props)) {
        n_cluster_artificial <- round(cluster_props[cluster] * n_artificial)
        
        if (n_cluster_artificial > 0) {
          genotype_cluster_cells <- real_cells[real_cells[[cluster_var]] == cluster, ]
          n_genotype_cluster <- nrow(genotype_cluster_cells)
          
          if (n_genotype_cluster >= min_cells_per_cluster && n_genotype_cluster > 0) {
            sampled_indices <- sample(1:n_genotype_cluster, n_cluster_artificial, replace = TRUE)
            artificial_cluster <- genotype_cluster_cells[sampled_indices, ]
            sampling_method <- "within_genotype"
            cat("  Cluster", cluster, ": sampling from genotype (n=", n_genotype_cluster, ")\n")
          } else {
            all_cluster_cells <- coords[coords[[cluster_var]] == cluster, ]
            if (nrow(all_cluster_cells) > 0) {
              sampled_indices <- sample(1:nrow(all_cluster_cells), n_cluster_artificial, replace = TRUE)
              artificial_cluster <- all_cluster_cells[sampled_indices, ]
              artificial_cluster[[genotype_var]] <- genotype
              sampling_method <- "full_umap"
              cat("  Cluster", cluster, ": sampling from full UMAP (n=", n_genotype_cluster, ")\n")
            } else {
              cat("  Cluster", cluster, ": no cells found in full UMAP, skipping\n")
              next
            }
          }
          
          artificial_cluster$cell_id <- paste0("artificial_", sampling_method, "_", 
                                              genotype, "_", cluster, "_", 
                                              1:nrow(artificial_cluster))
          artificial_cluster$sampling_method <- sampling_method
          
          if (nrow(artificial_cluster) > 0) {
            jitter1 <- rnorm(nrow(artificial_cluster), 0, jitter_amount)
            jitter2 <- rnorm(nrow(artificial_cluster), 0, jitter_amount)
            artificial_cluster$UMAP_1 <- artificial_cluster$UMAP_1 + jitter1
            artificial_cluster$UMAP_2 <- artificial_cluster$UMAP_2 + jitter2
          }
          
          artificial_cells_list[[cluster]] <- artificial_cluster
        }
      }
      
      if (length(artificial_cells_list) > 0) {
        artificial_cells <- do.call(rbind, artificial_cells_list)
        balanced_umap_list[[length(balanced_umap_list) + 1]] <- artificial_cells
      }
    }
  }
  
  balanced_umap <- do.call(rbind, balanced_umap_list)
  return(balanced_umap)
}

plot_sampling_facets_simple <- function(balanced_umap_data, 
                                       genotype_var = "condition",
                                       point_size = 0.3,
                                       alpha_value = 0.3) {
  
  ggplot(balanced_umap_data, aes(x = UMAP_1, y = UMAP_2, color = .data[[genotype_var]])) +
    geom_point(size = point_size, alpha = alpha_value) +
    facet_wrap(as.formula(paste("~", genotype_var))) +
    theme_classic() +
    labs(title = "Balanced UMAP Visualization")
}

In [ ]:
balanced_data1 <- create_balanced_umap_hybrid(seu, min_cells_per_cluster = 50, jitter_amount = 0.15)

plot_sampling_facets_simple(balanced_data1, point_size = 0.1, alpha = 0.1)

jpeg(file.path(figDir, "balanced_umap.jpeg"), width = 1333, height = 1067, res = 150)
print(plot_sampling_facets_simple(balanced_data1, point_size = 0.1, alpha = 0.1))
dev.off()

In [ ]:
### Run MAGIC
library(reticulate)
DefaultAssay(seu) <- "SCT"
use_condaenv("magic2", conda = "/data/ebaird/miniconda3/bin/conda", required = TRUE)
use_python("/data/ebaird/miniconda3/envs/magic2/bin/python", required = TRUE)
library(Rmagic)
mag <- Rmagic::magic(seu, verbose=F)
seu@assays$MAGIC_SCT <- mag@assays$MAGIC_SCT

In [ ]:
# save(seu, file=paste0(repDir, "magicRDS.RData"))
load(file=paste0(mainDir, "composition_DEG_signatures/magicRDS.RData"))

In [ ]:
# Bulk RNAseq candidates - using ALL filtered genes
bulkrnaseq <- list()
bulkrnaDir <- "/data/ebaird/scRNAseq/Baird.et.al.2025/RNAandATAC/"
diff_name <- "o1.lrt.T0.GAL.NDE.wRi.vs.T0.GAL.NDE.prosRi"
diff_name_flipped <- paste(rev(strsplit(diff_name, ".vs.")[[1]]), collapse=".vs.")
# Read RNA-seq data
tmp <- read.csv(paste0(bulkrnaDir, diff_name, ".csv"), row.names = 1)
tmp$gene <- sapply(strsplit(rownames(tmp), "_"), `[`, 2)

# With logFC filtering
bulkrnaseq[[diff_name]] <- tmp$gene[tmp$FDR < 0.05 & tmp$logFC > 0.5]
bulkrnaseq[[diff_name_flipped]] <- tmp$gene[tmp$FDR < 0.05 & tmp$logFC < -0.5]

seu <- AddModuleScore(object = seu,features = bulkrnaseq, name = paste0(names(bulkrnaseq),"_ams"),assay='MAGIC_SCT')
colnames(seu@meta.data)[which(regexpr("_ams",colnames(seu@meta.data))>0)] <- names(bulkrnaseq)
seu[['MAGIC_SCT_AddModuleScore']]<-CreateAssayObject(t(seu@meta.data[,names(bulkrnaseq)]))

scale_factor <- 300 / 100

# RNAseq plots
DefaultAssay(seu) <- "MAGIC_SCT_AddModuleScore"
tmp <- names(bulkrnaseq)

# Collect all FeaturePlots in a flat list
all_plots <- list()

for (gene_name in tmp) {
  cat(gene_name, "\t")
  feature_name <- gsub('_', '-', gene_name)
  
  # split.by returns a list of plots
  plots <- FeaturePlot(
    seu,
    features = feature_name,
    reduction = "umap",
    pt.size = 0.5,
    combine = FALSE,
    order = TRUE,
    split.by = "condition",
    label = FALSE,
    max.cutoff = "q99"
  )
  
  # Add color scale and hide legends for each plot
  plots <- lapply(plots, function(p) {
    p + scale_colour_gradientn(colours = cols) +
      theme(plot.title = element_text(hjust = 0.5, size = 10),
            legend.position = "none")
  })
  
  # Combine per gene horizontally and add gene title
  gene_wrap <- wrap_plots(plots, ncol = length(plots)) +
    plot_annotation(title = gene_name)
  
  all_plots <- c(all_plots, list(gene_wrap))
}

# Add DimPlot at the end
all_plots <- c(all_plots, list(
  DimPlot(seu, reduction = "umap", pt.size = 0.5, group.by = "seurat_clusters", split.by = "condition") +
    theme(legend.position = "none")  # hide legend for consistency
))

# Combine all gene plots vertically
combined <- wrap_plots(all_plots, ncol = 1) &
  theme(legend.position = "right")  # add single shared legend

# Export
ggexport(
  combined,
  filename = paste0(figDir, diff_name, ".UMAP.jpeg"),
  width = 3000 * scale_factor,
  height = 1800 * scale_factor,
  res = 300
)

# # Save processed data
# save(bulkrnaseq, file = paste0(repDir, 'RNAseq_o3_candidates_all_genes.RData'))

In [ ]:
# # Bulk ATACseq candidates - using ALL filtered genes
bulkatacseq <- list()
atacDir <- "/data/ebaird/scRNAseq/Baird.et.al.2025/RNAandATAC/"
# tmp_atac <- read.csv(paste0(refsDir, "/ATAC.4.Ethan.T0.GAL.vs.T0.FLP.DB.homer.anno.csv"))
tmp_atac <- data.frame(read.table(paste0(atacDir, 'ATAC.2026.gsea.gmx.txt'), sep='\t', header=TRUE))


# With FDR filtering
bulkatacseq$Flp.NvsFlp.pros.OPEN_inFlp.N.FDR0.05 <- tmp_atac$Gene[tmp_atac$FDR < 0.05 & tmp_atac$Fold > 0]
bulkatacseq$Flp.NvsFlp.pros.CLOSED_inFlp.N.FDR0.05 <- tmp_atac$Gene[tmp_atac$FDR < 0.05 & tmp_atac$Fold < 0]

# # Convert FlyBase IDs to gene symbols
# library(AnnotationDbi)
# library(org.Dm.eg.db)

# bulkatacseq <- lapply(bulkatacseq, function(ids) {
#   symbols <- mapIds(
#     org.Dm.eg.db,
#     keys = ids,
#     column = "SYMBOL",
#     keytype = "FLYBASE",
#     multiVals = "first"
#   )
#   symbols[!is.na(symbols)]
# })

seu <- AddModuleScore(object = seu,features = bulkatacseq, name = paste0(names(bulkatacseq),"_ams"),assay='MAGIC_SCT')
colnames(seu@meta.data)[which(regexpr("_ams",colnames(seu@meta.data))>0)] <- names(bulkatacseq)
seu[['MAGIC_SCT_AddModuleScore']]<-CreateAssayObject(t(seu@meta.data[,names(bulkatacseq)]))

scale_factor <- 300 / 100

# ATACseq plots
tmp <- names(bulkatacseq)
DefaultAssay(seu) <- "MAGIC_SCT_AddModuleScore"
g <- lapply(tmp, function(i) {
        cat(i, "\t")
        FeaturePlot(seu, reduction='umap',features=gsub('_','-',i), pt.size=0.5, combine=TRUE, order=TRUE,split.by='condition', label=TRUE, max.cutoff = "q99") & scale_colour_gradientn(colours =cols)
})
g[[length(g)+1]] <- DimPlot(seu, reduction='umap', pt.size=0.5, group.by="seurat_clusters",split.by='condition')
ggexport(ggarrange(plotlist = g, nrow = 3, ncol = 1), filename = paste0(figDir, "UMAP.bulkATACseq.contrast15.grhG4.NvsgrhG4.prosRi.jpeg"), width=4000*scale_factor, height=1800*scale_factor, res=300)

# save(bulkatacseq, file = paste0(repDir, 'ATAC_candidates_all_genes_', format(Sys.Date(), "%Y%m%d"), '.RData'))

In [ ]:
# -----------------------------
# Bulk ATACseq candidates
# -----------------------------

bulkatacseq <- list()
atacDir <- "/data/ebaird/scRNAseq/Baird.et.al.2025/RNAandATAC/"

tmp_atac <- read.table(
  paste0(atacDir, "leading.edge.RNA.ATAC.05.03.2026.csv"),
  sep = ",",
  header = TRUE,
  stringsAsFactors = FALSE,
  fill = TRUE
)

# Convert each column to a gene list
bulkatacseq <- lapply(tmp_atac, function(x) {
  x <- na.omit(x)
  x[x != ""]
})

# -----------------------------
# Add module scores
# -----------------------------

seu <- AddModuleScore(
  object = seu,
  features = bulkatacseq,
  name = paste0(names(bulkatacseq), "_ams"),
  assay = "MAGIC_SCT"
)

# rename metadata columns to match gene set names
colnames(seu@meta.data)[
  which(regexpr("_ams", colnames(seu@meta.data)) > 0)
] <- names(bulkatacseq)

# create assay from module scores
seu[['MAGIC_SCT_AddModuleScore']] <- CreateAssayObject(
  t(seu@meta.data[, names(bulkatacseq)])
)

scale_factor <- 300 / 100


# -----------------------------
# ATAC UMAP plots
# -----------------------------

DefaultAssay(seu) <- "MAGIC_SCT_AddModuleScore"

tmp <- names(bulkatacseq)

for (gene_name in tmp) {

  cat(gene_name, "\n")

  feature_name <- gsub("_", "-", gene_name)

  # Generate FeaturePlots split by condition
  plots <- FeaturePlot(
    seu,
    features = feature_name,
    reduction = "umap",
    pt.size = 0.5,
    combine = FALSE,
    order = TRUE,
    split.by = "condition",
    label = FALSE,
    max.cutoff = "q99"
  )

  # Apply colour scale and formatting
  plots <- lapply(plots, function(p) {
    p +
      scale_colour_gradientn(colours = cols) +
      theme(
        plot.title = element_text(hjust = 0.5, size = 10),
        legend.position = "right"
      )
  })

  # Combine condition plots horizontally
  feature_row <- wrap_plots(plots, ncol = length(plots)) +
    plot_annotation(title = gene_name)

  # Cluster plot underneath
  cluster_plot <- DimPlot(
    seu,
    reduction = "umap",
    pt.size = 0.5,
    group.by = "seurat_clusters",
    split.by = "condition"
  ) +
    theme(legend.position = "none")

  combined <- wrap_plots(
    feature_row,
    cluster_plot,
    ncol = 1
  )

  # Export
  ggexport(
    combined,
    filename = paste0(figDir, gene_name, ".leadingedge.UMAP.jpeg"),
    width = 3000 * scale_factor,
    height = 1200 * scale_factor,
    res = 300
  )
}

In [ ]:
# Jasper motif enrichment candidates
jasper_motif_candidates <- list()
jasper_dir <- "/data/vtheodorou/jaspar_tfbs_extraction/jaspar_enrichment/bin/data.files/"
contrast = "o5.ATAC.GAL.N.vs.FLP.N.FDR.sorted.concA"
pattern <- "/allEnrichments.tsv"
jasper <- read.table(
  paste0(jasper_dir, contrast, pattern),
  sep = "\t",
  header = TRUE,
  stringsAsFactors = FALSE
)

# Top 20 based on column pValueLog
top_jasper <- jasper[order(jasper$pValueLog, decreasing = TRUE), ][1:20, ]
jasper_motif_candidates[[contrast]] <- top_jasper$motifID
Idents(seurat_subsets$gal) <- "seurat_clusters"
# Dotplot of top motifs
jpeg(paste0(figDir, contrast, 'Gal.top20TFmotifs.JASPAR.dotploto5.jpeg'), quality = 100, width = 2000, height = 1000, res = 150)
print(
  DotPlot(seurat_subsets$gal, features = top_jasper$TF_name, dot.scale = 6) + 
  RotatedAxis() +
  theme(
    axis.text.x = element_text(size = 8)
  ) +
  scale_color_gradientn(
    colours = c("white", "forestgreen"),
    limits = c(0, 1.5),
    oob = scales::squish
  ) +
  ggtitle(paste("Gal Top 20 JASPAR Motifs", contrast))
)
dev.off()

In [ ]:
DefaultAssay(seu)<-'MAGIC_SCT'

vdir <- paste0(figDir,'violin.MAGIC_SCT_AddModuleScore/')
bdir <- paste0(figDir,'boxplot.MAGIC_SCT_AddModuleScore/')
dir.create(paste0(figDir,'boxplot.MAGIC_SCT_AddModuleScore'))
dir.create(paste0(figDir,'violin.MAGIC_SCT_AddModuleScore'))

# violin plots per cluster and per current annotation, split by sample
pdf(paste0(vdir, "violin.MAGIC_SCT_AddModuleScore.cell_types.pdf"), width=7.5, height=5)
for(i in names(bulkrnaseq)){
    g1 <- ggplot(seu@meta.data, aes_string(x='seurat_clusters', y = i)) + geom_violin() + theme_bw() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))
    print(g1)
}
dev.off()

# pdf(paste0(vdir, "violin.MAGIC_SCT_AddModuleScore.cell_types.pdf"), width=7.5, height=5)
# for(i in names(bulkatacseq)){
#     g1 <- ggplot(seu@meta.data, aes_string(x='seurat_clusters', y = i)) + geom_violin() + theme_bw() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))
#     print(g1)
# }
# dev.off()

#group.cols<-list(
#    2198='#756bb1',
#    2197='#bcbddc',
#    2196='#de2d26',
#    2199='#fc9272')

for(i in names(bulkrnaseq)){
    df <- data.frame(expression = as.numeric(seu@assays$MAGIC_SCT_AddModuleScore@data[gsub('_','-',i),]), condition = seu$condition,cluster=seu$seurat_clusters)
    png(paste0(bdir ,i,".png"), width=2000/1.75, height=3000/1.75)
    g1 <- ggplot(df, aes(x=condition, y = expression))+ geom_violin() +geom_jitter()  + theme_bw() + ggtitle(i)+facet_wrap(~cluster,ncol=4)+ theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))#+ scale_fill_manual(values = group.cols)
    print(g1)
    dev.off()
}

# for(i in names(bulkatacseq)){
#     df <- data.frame(expression = as.numeric(seu@assays$MAGIC_SCT_AddModuleScore@data[gsub('_','-',i),]), condition = seu$condition,cluster=seu$seurat_clusters)
#     png(paste0(bdir, i,".png"), width=2000/1.75, height=3000/1.75)
#     g1 <- ggplot(df, aes(x=condition, y = expression))+ geom_violin() +geom_jitter()  + theme_bw() + ggtitle(i)+facet_wrap(~cluster,ncol=4)+ theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))#+ scale_fill_manual(values = group.cols)
#     print(g1)
#     dev.off()
# }

In [ ]:
seu

In [ ]:
DefaultAssay(seu) <- "MAGIC_SCT"

In [ ]:
### Signatures

### Drosophila signatures
# sig_path <- file.path(refsDir, "gene_sets_wide.csv")
# signatures_df <- read.csv(sig_path, header = TRUE, stringsAsFactors = FALSE, check.names = FALSE)
# # Overwrite miRNA signature with miRNA present in current dataset
# all_micro_rnas <- rownames(seu)[grepl("mir-", rownames(seu))]
# all_micro_rnas
# signatures_df$miRNA_genes <- c(all_micro_rnas, rep(NA, nrow(signatures_df) - length(all_micro_rnas)))
# base_sig_dir <- file.path(figDir, "drosophila_signatures")
# dir.create(base_sig_dir, showWarnings = FALSE, recursive = TRUE)

# sig_path <- file.path(refsDir, "drosophila_ion_channel_genes.tsv")
# signatures_df <- read.csv(sig_path, header = TRUE, stringsAsFactors = FALSE, check.names = FALSE, sep="\t")
# base_sig_dir <- file.path(figDir, "drosophila_ion_channel_genes")
# dir.create(base_sig_dir, showWarnings = FALSE, recursive = TRUE)

# Human to fly signatures
# sig_path <- file.path("/data/ebaird/scRNAseq/refs/human_to_fly_signatures_high_mod_nobm.csv")
# signatures_df <- read.csv(sig_path, header = TRUE, stringsAsFactors = FALSE, check.names = FALSE)
# base_sig_dir <- file.path(figDir, "human_to_fly_signatures_high_mod_nobm")
# dir.create(base_sig_dir, showWarnings = FALSE, recursive = TRUE)

### Flybase signatures
# sig_path <- file.path(refsDir, "flybase_spliceosome_complexes_signatures.csv")
# signatures_df <- read.csv(sig_path, header = TRUE, stringsAsFactors = FALSE, check.names = FALSE)
# base_sig_dir <- file.path(figDir, "flybase_spliceosome_complexes_signatures")
# dir.create(base_sig_dir, showWarnings = FALSE, recursive = TRUE)

# signatures <- lapply(signatures_df, function(col) {
#   genes <- as.character(col)
#   genes <- genes[!is.na(genes) & genes != ""]
#   unique(genes)
# })

# Or only selected signatures
# selected_sigs <- c("neftel_NPC1", "neftel_NPC2", "neftel_OPC", "neftel_AC", "neftel_MES1", "neftel_MES2", "infiltrating_periphery", "beck_Cycling", "beck_Intermediate", "beck_Neuron.like", "beck_NSC.like")
# selected_sigs <- c("Idh_mut")
# signatures <- signatures[selected_sigs]
# manual signature
signatures <- list(
  # up_prosRi_bulk = c("ase", "HGTX", "oc", "disco", "ey", "sca", "bi", "toy", "pros"),
  # down_prosRi_bulk = c("btd", "Eip93F", "Ldh", "Dll", "GstD3", "Vdup1"),
  # test = c("pros", "elav", "tap", "disco", "bi", "futsch", "ase", "Hey", "cas"),
  # temporal_maurange = c("chinmo", "Imp", "Syp", "lin-28", "br", "Eip93F", "D", "Grh", "kek1", "spi", "rho", "rl"),
  # oscillatory_TF_QNSC = c("dpn", "klu", "wor", "pros", "Optix"),
  # xenium_cluster13 = c("retn", "Ldh", "Nox", "lncRNA:CR30009", "Delta", "tll", "wdp", "CycE", "nab", "elav", "wor", "ogre")
  rnai = c("HmgZ", "Eip93F", "pros", "Syp", "Bre1")
)
base_sig_dir <- file.path(figDir, "custom_signatures")
dir.create(base_sig_dir, showWarnings = FALSE, recursive = TRUE)

safe_name <- function(x) {
  x2 <- gsub("[^A-Za-z0-9_]", "_", x)
  x2 <- gsub("_+", "_", x2)
  x2 <- gsub("^_|_$", "", x2)
  if (nchar(x2) == 0) x2 <- paste0("sig_", sample(10000,1))
  make.names(x2)
}

has_genotype <- "genotype" %in% colnames(seu@meta.data)

for (orig_name in names(signatures)) {

  genes_all <- signatures[[orig_name]]
  if (length(genes_all) == 0) next

  present_genes <- intersect(genes_all, rownames(seu))
  if (length(present_genes) == 0) next

  sig_safe <- safe_name(orig_name)
  n_clusters <- length(unique(seu$seurat_clusters))
  sig_dir <- file.path(base_sig_dir, sig_safe)
  dir.create(sig_dir, showWarnings = FALSE, recursive = TRUE)

  ## =========================
  ## Module score
  ## =========================
  mod_name_prefix <- paste0(sig_safe, "_mod")
  seu <- AddModuleScore(
    seu,
    features = list(present_genes),
    name = mod_name_prefix
  )

  score_col <- paste0(mod_name_prefix, "1")

  ## =========================
  ## Module score FeaturePlot (split.by genotype)
  ## =========================
  if (score_col %in% colnames(seu@meta.data) && has_genotype) {

    plots <- FeaturePlot(
      seu,
      features = score_col,
      split.by = "genotype",
      order = TRUE,
      pt.size = 0.5,
      max.cutoff = "q99",
      keep.scale = "all",
      combine = FALSE
    )

    plots <- lapply(plots, function(p) {
      p +
        scale_colour_gradientn(colours = cols) +
        theme(
          legend.position = "right",
          plot.title = element_text(size = 10)
        )
    })

    p_mod <- patchwork::wrap_plots(
      plots,
      ncol = length(plots)
    )

    out_mod <- file.path(sig_dir, "module_score_split.jpeg")
    jpeg(
      out_mod,
      width = 600 * length(plots),
      height = 550,
      res = 150
    )
    print(p_mod)
    dev.off()
  }

  ## =========================
  ## Module score full dataset
  ## =========================
  p_mod2 <- FeaturePlot(
    seu,
    features = score_col,
    order = TRUE,
    max.cutoff = "q99"
  ) &
    scale_colour_gradientn(colours = cols)

  out_mod2 <- file.path(sig_dir, "module_score_full_dataset.jpeg")
  jpeg(out_mod2, width = 600, height = 550, res = 150)
  print(p_mod2)
  dev.off()

  ## =========================
  ## Individual gene UMAPs (split.by genotype)
  ## =========================
  if (has_genotype) {

    gene_plots <- lapply(present_genes, function(gene) {

      plots <- FeaturePlot(
        seu,
        features = gene,
        split.by = "condition",
        order = TRUE,
        pt.size = 0.5,
        max.cutoff = "q99",
        keep.scale = "all",
        combine = FALSE
      )

      plots <- lapply(plots, function(p) {
        p + theme(plot.title = element_text(hjust = 0.5, size = 10)) + 
          scale_colour_gradientn(colours = cols)
      })

      (
        patchwork::wrap_plots(
          plots,
          ncol = length(plots)
        ) +
          plot_annotation(title = gene)
      ) &
        theme(
          legend.position = "right",
          plot.title = element_text(hjust = 0.5, size = 10)
        )
    })

    chunk_size <- 40
    gene_chunks <- split(
      gene_plots,
      ceiling(seq_along(gene_plots) / chunk_size)
    )

    for (i in seq_along(gene_chunks)) {

      chunk_plots <- gene_chunks[[i]]

      out_genes_umap <- file.path(
        sig_dir,
        paste0("UMAP_genes_per_sample_part", i, ".jpeg")
      )

      jpeg(
        out_genes_umap,
        width = 700 * length(unique(seu$condition)),
        height = 550 * length(chunk_plots),
        res = 150
      )

      # DO NOT collect guides here
      combined <- patchwork::wrap_plots(
        chunk_plots,
        ncol = 1
      )

      print(combined)
      dev.off()
    }
  }


  ## =========================
  ## Individual gene UMAPs (full dataset)
  ## =========================
  gene_plots <- lapply(present_genes, function(gene) {
    FeaturePlot(
      seu,
      features = gene,
      order = TRUE,
      pt.size = 0.5,
      max.cutoff = "q99"
    ) &
      scale_colour_gradientn(colours = cols) &
      ggtitle(gene) &
      theme(plot.title = element_text(hjust = 0.5, size = 10))
  })

  chunk_size <- 40
  gene_chunks <- split(
    gene_plots,
    ceiling(seq_along(gene_plots) / chunk_size)
  )

  for (i in seq_along(gene_chunks)) {

    chunk_plots <- gene_chunks[[i]]
    ncol_grid <- 3
    nrow_grid <- ceiling(length(chunk_plots) / ncol_grid)

    out_genes_umap <- file.path(
      sig_dir,
      paste0("UMAP_genes_full_dataset_part", i, ".jpeg")
    )

    png(
      out_genes_umap,
      width = 2200,
      height = 650 * nrow_grid,
      res = 150
    )

    grobs_list <- lapply(chunk_plots, ggplotGrob)
    do.call(
      grid.arrange,
      c(grobs = grobs_list, ncol = ncol_grid)
    )
    dev.off()
  }

  ## =========================
  ## DotPlot (UNCHANGED)
  ## =========================
  genotype_colors <- c("blue", "red", "green", "purple", "orange", "brown")

  if (exists("seurat_subsets")) {

    dp_list <- lapply(seq_along(seurat_subsets), function(i) {

      g <- names(seurat_subsets)[i]
      seu_sub <- seurat_subsets[[g]]

      DotPlot(
        seu_sub,
        features = present_genes,
        assay = "RNA"
      ) +
        RotatedAxis() +
        ggtitle(paste0("DotPlot: ", orig_name, " (", g, ")")) +
        scale_colour_gradientn(
          colours = c("white", genotype_colors[i])
        ) +
        theme(plot.title = element_text(size = 10))
    })

    dp_combined <- patchwork::wrap_plots(dp_list, ncol = 1)

    plot_width <- max(8, 0.3 * length(present_genes))
    plot_height <- max(
      4,
      0.3 * n_clusters * length(seurat_subsets)
    )

    out_dot <- file.path(sig_dir, "DotPlot_combined.jpeg")

    ggsave(
      out_dot,
      plot = dp_combined,
      width = plot_width,
      height = plot_height,
      dpi = 300
    )

  } else {

    dp <- DotPlot(
      seu,
      features = present_genes,
      assay = "RNA"
    ) +
      RotatedAxis() +
      ggtitle(paste0("DotPlot: ", orig_name)) +
      scale_colour_gradientn(colours = c("white", "green")) +
      theme(plot.title = element_text(size = 10))

    plot_width <- max(6, 0.6 * length(present_genes))
    plot_height <- max(4, 0.5 * n_clusters * 2)

    out_dot <- file.path(sig_dir, "DotPlot.jpeg")

    ggsave(
      out_dot,
      plot = dp,
      width = plot_width,
      height = plot_height,
      dpi = 300
    )
  }


  # Heatmap
  max_heat_genes <- 200
  heat_genes <- head(present_genes, max_heat_genes)
  heat_genes <- unique(heat_genes)

  if (exists("seurat_subsets") && length(seurat_subsets) > 1) {
    library(patchwork)
    dp_list <- lapply(names(seurat_subsets), function(g) {
      seu_sub <- seurat_subsets[[g]]
      DoHeatmap(seu_sub, features = heat_genes, size = 3, assay = "SCT") +
        ggtitle(paste0("Heatmap: ", orig_name, " (", g, ")")) +
        guides(color = guide_colorbar(title = "Expression")) +
        theme(legend.position = "right")
    })
    
    ht_combined <- patchwork::wrap_plots(dp_list, ncol = 1)
    
    out_heat <- file.path(sig_dir, "Heatmap_combined.jpeg")
    heat_h <- 3 + 0.1 * length(heat_genes) * length(dp_list)
    ggsave(out_heat, plot = ht_combined, width = 10, height = heat_h, dpi = 300)
    message("Saved combined heatmap: ", out_heat)
    
  } else {
    ht <- DoHeatmap(seu, features = heat_genes, size = 3, assay = "SCT") +
      ggtitle(paste0("Heatmap: ", orig_name)) +
      guides(color = guide_colorbar(title = "Expression")) +
      theme(legend.position = "right")
    
    out_heat <- file.path(sig_dir, "Heatmap.jpeg")
    heat_h <- 3 + 0.1 * length(heat_genes)
    ggsave(out_heat, plot = ht, width = 10, height = heat_h, dpi = 300)
    message("Saved heatmap: ", out_heat)
  }
}

In [ ]:
# Function to add fly gene signatures directly (no ortholog conversion needed)
add_fly_signature_to_csv <- function(fly_genes, sig_name, csv_path = "fly_signatures.csv") {
  fly_genes <- unique(as.character(fly_genes))
  
  if (!file.exists(csv_path)) {
    sig_df <- data.frame(matrix(fly_genes, ncol = 1))
    colnames(sig_df) <- sig_name
    write.csv(sig_df, csv_path, row.names = FALSE)
    message("Created new CSV and added signature '", sig_name, "' with ", length(fly_genes), " fly genes")
  } else {
    sig_df <- read.csv(csv_path, check.names = FALSE)
    max_rows <- max(nrow(sig_df), length(fly_genes))
    
    if (nrow(sig_df) < max_rows) {
      sig_df[(nrow(sig_df) + 1):max_rows, ] <- NA
    }
    
    new_col <- c(fly_genes, rep(NA, max_rows - length(fly_genes)))
    sig_df[[sig_name]] <- new_col
    
    write.csv(sig_df, csv_path, row.names = FALSE)
    message("Added signature '", sig_name, "' with ", length(fly_genes), " fly genes")
  }
  
  return(fly_genes)
}

flybase_sigs_tsv_path <- paste0(refsDir, 'flybase_spliceosome_complexes/')
csvout <- paste0(refsDir, "flybase_spliceosome_complexes_signatures.csv")
tsv_files <- list.files(flybase_sigs_tsv_path, pattern = "\\.tsv$", full.names = TRUE)

for (tsv_file in tsv_files) {
  sig_name <- tools::file_path_sans_ext(basename(tsv_file))
  
  flybase_genes <- read.table(tsv_file, header = TRUE, sep = "\t")
  
  if ("Symbol" %in% colnames(flybase_genes)) {
    flybase_genes <- flybase_genes$Symbol
  } else if ("Gene.Symbol" %in% colnames(flybase_genes)) {
    flybase_genes <- flybase_genes$Gene.Symbol
  } else if (ncol(flybase_genes) == 1) {
    flybase_genes <- flybase_genes[[1]]
  } else {
    warning("Could not identify gene symbol column in ", tsv_file, ". Using first column.")
    flybase_genes <- flybase_genes[[1]]
  }
  
  flybase_genes <- unique(na.omit(as.character(flybase_genes)))
  
  if (length(flybase_genes) > 0) {
    add_fly_signature_to_csv(flybase_genes, sig_name, csvout)
  } else {
    warning("No genes found for signature '", sig_name, "' after filtering")
  }
}

# Add manual genes
genes <- c("Idh", "Got1", "Got2", "Gdh", "Ldh")

genes <- intersect(genes, rownames(seu))
genes

csvout <- paste0(refsDir, "gene_sets_wide.csv")

add_fly_signature_to_csv(genes, "Idh_mut", csvout)

In [ ]:
library(reticulate)
use_python("/data/ebaird/miniconda3/envs/TCGA/bin/python3")
diopt <- import("pyDIOPT")

In [ ]:
# Reverse the DIOPT function to convert human to fly genes
Hs_to_Dm_diopt_batch <- function(human_genes, batch_size = 500, filter_method = "best_match", max_retries = 3, retry_delay = 5, timeout_seconds = 60) {
  release <- diopt$DIOPTRelease("v9", "h sapiens")
  
  # Split into batches to avoid timeouts
  batches <- split(human_genes, ceiling(seq_along(human_genes)/batch_size))
  
  results <- list()
  
  for (i in seq_along(batches)) {
    cat("Processing batch", i, "of", length(batches), "- Genes:", length(batches[[i]]), "\n")
    
    batch_result <- NULL
    retry_count <- 0
    
    while (is.null(batch_result) && retry_count < max_retries) {
      tryCatch({
        # Add timeout protection
        result <- NULL
        start_time <- Sys.time()
        
        # Execute with timeout protection - note the target_species change
        result <- release$fetch(
          as.list(batches[[i]]),
          target_species = "d melanogaster",  # Changed target to fly
          filter = "best_match",
          condensed = FALSE
        )
        
        # Check if we exceeded timeout
        if (as.numeric(Sys.time() - start_time) > timeout_seconds) {
          stop("Request timed out after ", timeout_seconds, " seconds")
        }
        
        batch_result <- py_to_r(result)
        cat("  Success - found", nrow(batch_result), "orthologs\n")
        
      }, error = function(e) {
        retry_count <<- retry_count + 1
        if (retry_count < max_retries) {
          cat("  Error (attempt ", retry_count, "/", max_retries, "): ", e$message, "\n", sep = "")
          cat("  Waiting", retry_delay, "seconds before retry...\n")
          Sys.sleep(retry_delay)
        } else {
          cat("  Failed after", max_retries, "attempts:", e$message, "\n")
        }
      })
    }
    
    if (!is.null(batch_result)) {
      results[[i]] <- batch_result
    } else {
      cat("  Skipping batch", i, "after", max_retries, "failed attempts\n")
      # Add empty result to maintain batch order
      results[[i]] <- data.frame()
    }
    
    # Add delay between batches to be respectful to the server
    if (i < length(batches)) {
      Sys.sleep(1)
    }
  }
  
  # Combine all results (including empty dataframes for failed batches)
  if (length(results) > 0) {
    # Filter out completely NULL results and bind rows
    valid_results <- results[sapply(results, function(x) !is.null(x) && nrow(x) > 0)]
    
    if (length(valid_results) > 0) {
      final_result <- do.call(rbind, valid_results)
      cat("\n=== COMPLETED ===\n")
      cat("Successfully processed batches:", length(valid_results), "/", length(batches), "\n")
      cat("Total orthologs found:", nrow(final_result), "\n")
      return(final_result)
    } else {
      cat("\n=== FAILED ===\n")
      cat("No batches were successfully processed\n")
      return(NULL)
    }
  } else {
    return(NULL)
  }
}

# Updated function to get fly orthologs using DIOPT with best_match + high rank filtering
add_signature_to_csv <- function(human_genes, sig_name, csv_path = "fly_signatures_diopt.csv") {
  # Get ortholog mapping using DIOPT with best_match filter
  ortho_results <- Hs_to_Dm_diopt_batch(unique(human_genes))
  
  if (is.null(ortho_results) || nrow(ortho_results) == 0) {
    warning("No orthologs found for signature '", sig_name, "'")
    fly_genes <- character(0)
  } else {
    # Extract fly genes from DIOPT results with additional high-rank filtering
    ortho_map <- ortho_results %>%
      as_tibble() %>%
      dplyr::select(
        human_gene = input_symbol, 
        fly_gene = target_symbol,
        rank = target_confidence  # Add rank column for filtering
      ) %>%
      filter(!is.na(fly_gene)) %>%
      # ADDITIONAL FILTER: Keep only high-rank orthologs
      filter(rank %in% c("high", "moderate")) %>%
      distinct(human_gene, fly_gene, .keep_all = TRUE)
    
    fly_genes <- ortho_map %>%
      pull(fly_gene) %>%
      unique()
    
    # Report statistics
    total_orthologs <- ortho_results %>% filter(!is.na(target_symbol)) %>% nrow()
    high_rank_orthologs <- nrow(ortho_map)
    # cat("  DIOPT best_match found: ", total_orthologs, " orthologs\n", sep = "")
    cat("  After high/moderate-rank filtering: ", high_rank_orthologs, " orthologs\n", sep = "")
    cat("  Final unique fly genes: ", length(fly_genes), "\n", sep = "")
  }
  
  # Check if CSV exists
  if (!file.exists(csv_path)) {
    # Create new CSV with this signature as the first column
    sig_df <- data.frame(matrix(fly_genes, ncol = 1))
    colnames(sig_df) <- sig_name
    write.csv(sig_df, csv_path, row.names = FALSE)
    message("Created new CSV and added signature '", sig_name, "' with ", length(fly_genes), " fly orthologs")
  } else {
    # Load existing CSV
    sig_df <- read.csv(csv_path, check.names = FALSE)
    
    # Add new signature column
    max_rows <- max(nrow(sig_df), length(fly_genes))
    
    # Extend existing data frame if needed
    if (nrow(sig_df) < max_rows) {
      sig_df[(nrow(sig_df) + 1):max_rows, ] <- NA
    }
    
    # Create the new column with proper length
    new_col <- c(fly_genes, rep(NA, max_rows - length(fly_genes)))
    sig_df[[sig_name]] <- new_col
    
    # Write updated CSV
    write.csv(sig_df, csv_path, row.names = FALSE)
    message("Added signature '", sig_name, "' with ", length(fly_genes), " fly orthologs")
  }

  # Record genes lost during mapping in a csv with signature name it came from as column name
  if (exists("ortho_map")) {
    lost_genes <- setdiff(unique(human_genes), ortho_map$human_gene)
  } else {
    lost_genes <- unique(human_genes)  # All genes lost if no ortho_map
  }
  
  if (length(lost_genes) > 0) {
    lost_csv_path <- paste0(refsDir, "human_to_fly_signatures_lost_genes_high_nobm.csv")
    if (!file.exists(lost_csv_path)) {
      lost_df <- data.frame(matrix(lost_genes, ncol = 1))
      colnames(lost_df) <- sig_name
      write.csv(lost_df, lost_csv_path, row.names = FALSE)
    } else {
      lost_df <- read.csv(lost_csv_path, check.names = FALSE)
      max_rows_lost <- max(nrow(lost_df), length(lost_genes))
      if (nrow(lost_df) < max_rows_lost) {
        lost_df[(nrow(lost_df) + 1):max_rows_lost, ] <- NA
      }
      new_lost_col <- c(lost_genes, rep(NA, max_rows_lost - length(lost_genes)))
      lost_df[[sig_name]] <- new_lost_col
      write.csv(lost_df, lost_csv_path, row.names = FALSE)
    }
    message("Recorded ", length(lost_genes), " lost human genes for signature '", sig_name, "'")
  }
  
  return(fly_genes)
}

# The rest of your workflow remains the same:
# Output csv path
csvout <- paste0(refsDir, "human_to_fly_signatures_high_nobm.csv")

# # Process csv or xlsx signatures
# sigs_path1 <- '/data/ebaird/scRNAseq/Nikos_BreRivsG4wRi/refs/verhaak.signatures.xlsx'
# sigs_path2 <- '/data/ebaird/scRNAseq/beck/Beck_signatures.csv'
# sigs_path3 <- '/data/ebaird/scRNAseq/refs/neftel.csv'
# signame1 <- tools::file_path_sans_ext(basename(sigs_path1))
# signame2 <- tools::file_path_sans_ext(basename(sigs_path2))
# signame3 <- tools::file_path_sans_ext(basename(sigs_path3))
# sigs1 <- readxl::read_excel(sigs_path1, sheet = 1)
# sigs2 <- read.csv(sigs_path2, sep = '\t')
# sigs3 <- read.csv(sigs_path3, sep = ',')
# sigs_list <- list(
#   verhaak = sigs1,
#   beck = sigs2,
#   neftel = sigs3
# )

# # Process each signature set in sigs_list
# for (sigset_name in names(sigs_list)) {
#   sigset <- sigs_list[[sigset_name]]
#   # For data frames with multiple columns (e.g., multiple lineages)
#   for (lineage_name in colnames(sigset)) {
#     human_genes <- sigset[[lineage_name]] %>% na.omit() %>% as.character()
#     if (length(human_genes) > 0) {
#       add_signature_to_csv(human_genes, paste0(sigset_name, "_", lineage_name), csvout)
#     }
#   }
# }

# Process manually curated signature

module18.wang.2021 <- c("PRRX2", "SETD2", "ABCC1", "ARID48", "KLF5", "INMT", "GSX2", "SP7", "TP63",
                        "CAD", "HAL", "GAD2", "GAD1", "PLOD2", "D2HGDH", "BCAT1", "MYT1L")

module6.wang.2021 <- c("CYP4X1", "EMX1", "GAD1", "GAD2", "MYT1L", "PXDN", "SLC27A2", "SLC4A10",
                       "ENTPD3", "CKMT1B", "CKMT1A") 

add_signature_to_csv(module6.wang.2021, "module6.wang.2021", csvout)

# Record in csv which genes from each signature not present in scRNA-seq dataset
absent_genes_csv <- paste0(refsDir, "human_to_fly_signatures_absent_genes_high_nobm.csv")
if (file.exists(csvout)) {
  sig_df <- read.csv(csvout, check.names = FALSE)
  absent_genes_list <- lapply(names(sig_df), function(sig_name) {
    sig_genes <- sig_df[[sig_name]]
    sig_genes <- sig_genes[!is.na(sig_genes)]
    absent_genes <- setdiff(sig_genes, rownames(seu))
    absent_genes
  })
  names(absent_genes_list) <- names(sig_df)
  
  # Only proceed if there are any absent genes at all
  if (any(sapply(absent_genes_list, length) > 0)) {
    max_rows_absent <- max(sapply(absent_genes_list, length))
    absent_df <- data.frame(matrix(NA, nrow = max_rows_absent, ncol = length(absent_genes_list)))
    colnames(absent_df) <- names(absent_genes_list)
    
    for (sig_name in names(absent_genes_list)) {
      absent_genes <- absent_genes_list[[sig_name]]
      # Only assign if there are absent genes
      if (length(absent_genes) > 0) {
        absent_df[1:length(absent_genes), sig_name] <- absent_genes
      }
    }
    
    write.csv(absent_df, absent_genes_csv, row.names = FALSE)
    message("Recorded absent genes per signature in ", absent_genes_csv)
  } else {
    message("No absent genes found for any signature.")
    # Create an empty file or skip creation
    write.csv(data.frame(), absent_genes_csv, row.names = FALSE)
  }
} else {
  warning("Signatures CSV not found at ", csvout, ". Cannot record absent genes.")
}

In [ ]:
# Search for gene in seu TRUE or FALSE
"l(1)sc" %in% rownames(seu)

In [ ]:
# Read the files
all_fly_orthologs_csv <- read.csv(paste0(refsDir, "human_to_fly_signatures_high_nobm.csv"), check.names = FALSE)
lost_genes_csv <- read.csv(paste0(refsDir, "human_to_fly_signatures_lost_genes_high_nobm.csv"), check.names = FALSE)
absent_genes_csv <- read.csv(paste0(refsDir, "human_to_fly_signatures_absent_genes_high_nobm.csv"), check.names = FALSE)

# Create a new CSV with only the genes present in the dataset
present_genes_csv <- data.frame(matrix(ncol = ncol(all_fly_orthologs_csv), nrow = 0))
colnames(present_genes_csv) <- colnames(all_fly_orthologs_csv)

# Filter each signature to keep only genes present in the dataset
for (sig_name in colnames(all_fly_orthologs_csv)) {
  all_genes <- all_fly_orthologs_csv[[sig_name]]
  all_genes <- all_genes[!is.na(all_genes)]
  
  # Filter for genes present in the dataset
  present_genes <- intersect(all_genes, rownames(seu))
  
  # Add to the new present_genes_csv
  max_rows <- max(nrow(present_genes_csv), length(present_genes))
  
  # Extend the data frame if needed
  if (nrow(present_genes_csv) < max_rows) {
    present_genes_csv[(nrow(present_genes_csv) + 1):max_rows, ] <- NA
  }
  
  # Create the column with proper length
  new_col <- c(present_genes, rep(NA, max_rows - length(present_genes)))
  present_genes_csv[[sig_name]] <- new_col
}

# Save the present genes CSV
write.csv(present_genes_csv, paste0(refsDir, "human_to_fly_signatures_present_genes_high_nobm.csv"), row.names = FALSE)
message("Saved present genes to: ", paste0(refsDir, "human_to_fly_signatures_present_genes_high_nobm.csv"))

# Now create the statistics
signature_stats <- data.frame(
  Signature = character(),
  Present = integer(),
  Lost = integer(),
  Absent = integer(),
  stringsAsFactors = FALSE
)

for (sig_name in colnames(all_fly_orthologs_csv)) {
  # Get present genes (the ones we just filtered)
  present_genes <- present_genes_csv[[sig_name]]
  present_genes <- present_genes[!is.na(present_genes)]
  
  # Get lost genes (human genes that didn't map to fly orthologs)
  lost_genes <- if (sig_name %in% colnames(lost_genes_csv)) {
    lost_col <- lost_genes_csv[[sig_name]]
    lost_col[!is.na(lost_col)]
  } else {
    character(0)
  }
  
  # Get absent genes (fly orthologs not present in dataset)
  # These are the genes that were in all_fly_orthologs but not in present_genes
  all_fly_genes <- all_fly_orthologs_csv[[sig_name]]
  all_fly_genes <- all_fly_genes[!is.na(all_fly_genes)]
  absent_genes <- setdiff(all_fly_genes, present_genes)
  
  signature_stats <- rbind(signature_stats, data.frame(
    Signature = sig_name,
    Present = length(present_genes),
    Lost = length(lost_genes),
    Absent = length(absent_genes),
    stringsAsFactors = FALSE
  ))
}

# Create the plot
signature_stats_melt <- reshape2::melt(signature_stats, id.vars = "Signature", variable.name = "Status", value.name = "Count")

p_stats <- ggplot(signature_stats_melt, aes(x = Signature, y = Count, fill = Status)) +
  geom_bar(stat = "identity", position = "stack") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  ggtitle("Signature Gene Mapping Statistics") +
  ylab("Number of Genes") +
  xlab("Signature") +
  scale_fill_manual(values = c("Present" = "green", "Lost" = "orange", "Absent" = "red"))

ggsave(filename = paste0(figDir, "signature_mapping_statistics_diopt_high_nobm.jpeg"), plot = p_stats, width = 10, height = 6, dpi = 300)

# Print summary statistics
cat("\n=== Signature Mapping Summary ===\n")
for (i in 1:nrow(signature_stats)) {
  total_original <- signature_stats$Present[i] + signature_stats$Lost[i] + signature_stats$Absent[i]
  present_pct <- round(signature_stats$Present[i] / total_original * 100, 1)
  lost_pct <- round(signature_stats$Lost[i] / total_original * 100, 1)
  absent_pct <- round(signature_stats$Absent[i] / total_original * 100, 1)
  
  cat(signature_stats$Signature[i], ":\n")
  cat("  Present: ", signature_stats$Present[i], " (", present_pct, "%)\n", sep = "")
  cat("  Lost:    ", signature_stats$Lost[i], " (", lost_pct, "%)\n", sep = "")
  cat("  Absent:  ", signature_stats$Absent[i], " (", absent_pct, "%)\n", sep = "")
  cat("  Total:   ", total_original, "\n\n", sep = "")
}

In [ ]:
# Function for enrichment analysis

GO_KEGG_dir <- paste0(figDir, "GO/KEGG/")
dir.create(GO_KEGG_dir, showWarnings = FALSE, recursive = TRUE)
GO_BP_dir <- paste0(figDir, "GO/BP/")
dir.create(GO_BP_dir, showWarnings = FALSE, recursive = TRUE)
GO_CC_dir <- paste0(figDir, "GO/CC/")
dir.create(GO_CC_dir, showWarnings = FALSE, recursive = TRUE)
GO_MF_dir <- paste0(figDir, "GO/MF/")
dir.create(GO_MF_dir, showWarnings = FALSE, recursive = TRUE)
MSigDB_dir <- paste0(figDir, "MSigDB/")
dir.create(MSigDB_dir, showWarnings = FALSE, recursive = TRUE)

run_drosophila_enrichment <- function(gene_list, name) {
  fb_ids <- mapIds(org.Dm.eg.db,
                   keys = gene_list,
                   column = "FLYBASE",
                   keytype = "SYMBOL")
  
  fb_ids <- na.omit(fb_ids)

  ego <- enrichGO(gene = fb_ids,
                OrgDb = org.Dm.eg.db,
                keyType = "FLYBASE",
                ont = "BP",
                pAdjustMethod = "BH",
                qvalueCutoff = 0.05)
  
  as.data.frame(ego)
  if (is.null(ego) || nrow(ego) == 0) {
    cat("No significant GO terms found for", name, "\n")
    return(NULL)
  }

  write.csv(as.data.frame(ego), 
            paste0(tabDir, "GO_BP", name, ".csv"), 
            row.names = FALSE)
  
  if (nrow(ego) > 0) {
    dotplot(ego, showCategory=15) + 
      ggtitle(paste("GO Enrichment:", name))
    ggsave(paste0(GO_BP_dir, "GO_dotplot_BP_", name, ".png"), width=10, height=8)
  }

    ego <- enrichGO(gene = fb_ids,
                OrgDb = org.Dm.eg.db,
                keyType = "FLYBASE",
                ont = "CC",
                pAdjustMethod = "BH",
                qvalueCutoff = 0.05)
  
  as.data.frame(ego)
  if (is.null(ego) || nrow(ego) == 0) {
    cat("No significant GO terms found for", name, "\n")
    return(NULL)
  }

  write.csv(as.data.frame(ego), 
            paste0(tabDir, "GO_CC", name, ".csv"), 
            row.names = FALSE)
  
  if (nrow(ego) > 0) {
    dotplot(ego, showCategory=15) + 
      ggtitle(paste("GO Enrichment:", name))
    ggsave(paste0(GO_CC_dir, "GO_dotplot_CC_", name, ".png"), width=10, height=8)
  }

      ego <- enrichGO(gene = fb_ids,
                OrgDb = org.Dm.eg.db,
                keyType = "FLYBASE",
                ont = "MF",
                pAdjustMethod = "BH",
                qvalueCutoff = 0.05)
  
  as.data.frame(ego)
  if (is.null(ego) || nrow(ego) == 0) {
    cat("No significant GO terms found for", name, "\n")
    return(NULL)
  }

  write.csv(as.data.frame(ego), 
            paste0(tabDir, "GO_MF", name, ".csv"), 
            row.names = FALSE)
  
  if (nrow(ego) > 0) {
    dotplot(ego, showCategory=15) + 
      ggtitle(paste("GO Enrichment:", name))
    ggsave(paste0(GO_MF_dir, "GO_dotplot_MF_", name, ".png"), width=10, height=8)
  }
  
  entrez_ids <- mapIds(org.Dm.eg.db,
                       keys = gene_list,
                       column = "ENTREZID",
                       keytype = "SYMBOL",
                       multiVals = "first")

  # entrez_ids names as a list
  names(entrez_ids) <- entrez_ids
  entrez_ids <- na.omit(entrez_ids)

  # KEGG enrichment
  kegg <- enrichKEGG(gene = names(entrez_ids),
                     organism = "dme",
                     keyType = "ncbi-geneid",
                     pvalueCutoff = 0.05)
  
  as.data.frame(kegg)
  if (is.null(kegg) || nrow(ego) == 0) {
    cat("No significant GO terms found for", name, "\n")
    return(NULL)
  }

  if (!is.null(kegg) && nrow(kegg) > 0) {
    write.csv(as.data.frame(kegg), 
              paste0(tabDir, "KEGG_", name, ".csv"), 
              row.names = FALSE)
  }

  # Plot KEGG enrichment
  if (!is.null(kegg) && nrow(kegg) > 0) {
    dotplot(kegg, showCategory=15) + 
      ggtitle(paste("KEGG Enrichment:", name))
    ggsave(paste0(GO_KEGG_dir, "KEGG_dotplot_", name, ".png"), width=10, height=8)
  }

  # MSigDB hallmark gene sets
  C <- msigdbr(species = "Homo sapiens", 
              collection = "C2", 
              subcollection = "CGP") %>%
    dplyr::select(gs_name, gs_id, human_gene = gene_symbol) #%>%
    # dplyr::filter(gs_id %in% c('M2116', 'M2121', 'M2122', 'M2115'))

  # Map to orthologues
  ortho_map <- gorth(
    query = unique(C$human_gene),
    source_organism = "hsapiens",
    target_organism = "dmelanogaster",
    mthreshold = 1,
    filter_na = TRUE
  ) %>% 
    as_tibble() %>%
    dplyr::select(human_gene = input, fb_ids = ortholog_ensg) %>%
    mutate(fb_ids = mapIds(org.Dm.eg.db, 
                              keys = fb_ids, 
                              column = "FLYBASE", 
                              keytype = "ENSEMBL")) %>%
    na.omit()

  # Create fly hallmark gene sets
  fly_C <- C %>%
    inner_join(ortho_map, by = "human_gene") %>%
    dplyr::select(term = gs_name, gene = fb_ids) %>%
    distinct()

  C <- enricher(
    gene = fb_ids,
    universe = keys(org.Dm.eg.db, keytype = "FLYBASE"),
    TERM2GENE = fly_C,
    pvalueCutoff = 0.05,
    pAdjustMethod = "BH",
    qvalueCutoff = 0.2
  )

  print(C)

  if (!is.null(C) && nrow(C) > 0) {
      dotplot(C, showCategory=15) + 
        ggtitle(paste("CGP Enrichment:", name))
      ggsave(paste0(MSigDB_dir, "CGP_dotplot_", name, ".png"), width=10, height=8)
    }

}

In [ ]:
# Gene ontology/enrichment for DEGs between genotypes

de_files <- list.files(tabDir, pattern = "^DE_results_.*_vs_.*\\.csv$", full.names = TRUE)
de_results_list <- lapply(de_files, function(f) {
  if (file.info(f)$size == 0 || length(readLines(f, n = 10)) < 2) {
    pair <- gsub("^DE_results_(.*)\\.csv$", "\\1", basename(f))
    return(list(name = pair, up = character(0), down = character(0)))
  }
  df <- tryCatch(read.csv(f, row.names = 1), error = function(e) NULL)
  pair <- gsub("^DE_results_(.*)\\.csv$", "\\1", basename(f))
  if (is.null(df) || nrow(df) == 0) {
    return(list(name = pair, up = character(0), down = character(0)))
  }
  list(
    name = pair,
    up = rownames(df)[df$p_val < 0.05 & df$avg_log2FC > 0.5],
    down = rownames(df)[df$p_val < 0.05 & df$avg_log2FC < -0.5]
  )
})

all_markers <- read.csv(paste0("/data/ebaird/scRNAseq/SCENTINELsep24/QC_clustering/tables/allMarkers_merged_clusters.csv"), row.names = 1)
cluster_marker_lists <- lapply(sort(unique(all_markers$cluster)), function(cl) {
  all_markers$gene[all_markers$cluster == cl & all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]
})
names(cluster_marker_lists) <- paste0("cluster_", sort(unique(all_markers$cluster)))
allmarkers_full <- all_markers$gene[all_markers$p_val < 0.05 & abs(all_markers$avg_log2FC) > 0.5]

for (de in de_results_list) {
  if (length(de$up) > 0) run_drosophila_enrichment(de$up, paste0(de$name, "_up"))
  if (length(de$down) > 0) run_drosophila_enrichment(de$down, paste0(de$name, "_down"))
}
for (cl in names(cluster_marker_lists)) {
  run_drosophila_enrichment(cluster_marker_lists[[cl]], cl)
}
run_drosophila_enrichment(allmarkers_full, "all_markers")

In [ ]:
# saveRDS(seu, file = paste0(repDir, "signatures.rds"))
seu <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep24/composition_DEG_signatures/signatures.rds")

In [ ]:
### Convert to python anndata format
DefaultAssay(seu) <- "RNA"
use_condaenv("cellbender")
as.anndata(x = seu, file_path = "/data/ebaird/scRNAseq/ProsRivsG4wRi/composition_DEG_signatures/", file_name = "anndata.h5ad")

In [ ]:
counts_matrix <- GetAssayData(seu, assay = "RNA", slot = "counts")
metadata <- seu@meta.data

write.csv(metadata, 
          file = "/data/ebaird/scRNAseq/Baird.et.al.Nov25/composition_DEG_signatures/metadata.csv")

Matrix::writeMM(counts_matrix, 
                file = "/data/ebaird/scRNAseq/Baird.et.al.Nov25/composition_DEG_signatures/counts_matrix.mtx")

write.csv(data.frame(gene = rownames(counts_matrix)), 
          file = "/data/ebaird/scRNAseq/Baird.et.al.Nov25/composition_DEG_signatures/genes.csv")

write.csv(data.frame(barcode = colnames(counts_matrix)), 
          file = "/data/ebaird/scRNAseq/Baird.et.al.Nov25/composition_DEG_signatures/barcodes.csv")

In [ ]:
DefaultAssay(seu) <- "SCT"

pca_coords <- Embeddings(seu, reduction = "pca")
write.csv(pca_coords, paste0(repDir, "seurat_pca.csv"))

umap_coords <- Embeddings(seu, reduction = "umap")
write.csv(umap_coords, paste0(repDir, "seurat_umap.csv"))

In [ ]:
# Export highly variable genes
hvg_genes <- VariableFeatures(seu)

# Save to file
writeLines(hvg_genes, paste0(repDir, "seurat_hvg_genes.txt"))

In [ ]:
# Mapping new seurat to old seurat
seu24 <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep24/QC_clustering/merged_clusters.rds")
seu_prosRi <- readRDS("/data/ebaird/scRNAseq/ProsRivsG4wRi/QC_clustering/final_clusters.rds")

In [ ]:
# Load Dataset 2 markers
dataset2_markers <- read.csv("/data/ebaird/scRNAseq/ProsRivsG4wRi/QC_clustering/tables/allMarkers_final_clusters.csv")
dataset1 <- seu24

dataset2_clusters <- sort(unique(dataset2_markers$cluster))

for (cluster_num in dataset2_clusters) {
  # Get marker genes
  cluster_data <- dataset2_markers[dataset2_markers$cluster == cluster_num, ]
  cluster_data <- cluster_data[order(cluster_data$avg_log2FC, decreasing = TRUE), ]
  top_genes <- head(cluster_data$gene, 10)
  top_genes <- top_genes[top_genes %in% rownames(dataset1)]
  
  if (length(top_genes) >= 5) {
    # Add module score
    dataset1 <- AddModuleScore(dataset1, features = list(top_genes), name = paste0("cluster", cluster_num, "_prosRi_"))
    
    # Use the score for coloring in a simple plot
    feature_name <- paste0("cluster", cluster_num, "_prosRi_1")
    
    # Basic plot without complex features
    plot <- FeaturePlot(
      object = dataset1,
      features = feature_name,
      order = TRUE,  # Plot positive cells on top
      raster = FALSE  # Avoid rasterization issues
    ) & scale_colour_gradientn(colours = cols)
    
    print(plot)
  }
}